## Load Packages

In [ ]:
!pip install tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
import re
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
#from keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
import keras
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import math
import nltk

## Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path ="/content/drive/MyDrive/Colab Notebooks/IMDB Dataset.csv"

data = pd.read_csv(path)


In [ ]:
data.shape

(50000, 2)

In [ ]:
list(data.columns)

['review', 'sentiment']

In [ ]:
data.review[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [ ]:
data = data.sample(10000)
data.shape

(10000, 2)

## Pre-Processing

In [ ]:
def remove_tags(string):
    removelist = ""
    result = re.sub('','',string)          #remove HTML tags
    result = re.sub('https://.*','',result)   #remove URLs
    result = re.sub(r'[^w'+removelist+']', ' ',result)    #remove non-alphanumeric characters
    result = result.lower()
    return result
data['review']=data['review'].apply(lambda cw : remove_tags(cw))

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))
data['review'] = data['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop_words)]))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
nltk.download('wordnet')
w_tokenizer = nltk.tokenize.WhitespaceTokenizer()
lemmatizer = nltk.stem.WordNetLemmatizer()
def lemmatize_text(text):
    st = ""
    for w in w_tokenizer.tokenize(text):
        st = st + lemmatizer.lemmatize(w) + " "
    return st
data['review'] = data.review.apply(lemmatize_text)
#data

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
s = 0.0
for i in data['review']:
    word_list = i.split()
    s = s + len(word_list)
print("Average length of each review : ",s/data.shape[0])
pos = 0
for i in range(data.shape[0]):
    if data.iloc[i]['sentiment'] == 'positive':
        pos = pos + 1
neg = data.shape[0]-pos
print("Percentage of reviews with positive sentiment is "+str(pos/data.shape[0]*100)+"%")
print("Percentage of reviews with negative sentiment is "+str(neg/data.shape[0]*100)+"%")

Average length of each review :  18.7176
Percentage of reviews with positive sentiment is 49.79%
Percentage of reviews with negative sentiment is 50.21%


## Data Preparation: Label Encoding, Padding and Splitting

In [ ]:
reviews = data['review'].values
labels = data['sentiment'].values
encoder = LabelEncoder()
encoded_labels = encoder.fit_transform(labels)

In [ ]:
encoded_labels

array([1, 0, 1, ..., 1, 0, 1])

In [ ]:
train_sentences, test_sentences, train_labels, test_labels = train_test_split(reviews, encoded_labels, stratify = encoded_labels)

In [ ]:
vocab_size = 3000 # choose based on statistics
oov_tok = ''
embedding_dim = 100
max_length = 200 # choose based on statistics, for example 150 to 200
padding_type='post'
trunc_type='post'


In [ ]:
# tokenize sentences
tokenizer = Tokenizer(num_words = vocab_size, oov_token=oov_tok)
tokenizer.fit_on_texts(train_sentences)
word_index = tokenizer.word_index


In [ ]:
# convert train dataset to sequence and pad sequences
train_sequences = tokenizer.texts_to_sequences(train_sentences)
train_padded = pad_sequences(train_sequences, padding='post', maxlen=max_length)

# convert Test dataset to sequence and pad sequences
test_sequences = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_sequences, padding='post', maxlen=max_length)

## Model Building

In [ ]:
model = keras.Sequential()
model.add(keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length))
model.add(keras.layers.LSTM(64))
model.add(keras.layers.Dense(24, activation='relu'))
model.add(keras.layers.Dense(1, activation='sigmoid'))

model.build(input_shape=(None, max_length))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 100)       │       300,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 24)             │         1,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            25 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 343,825 (1.31 MB)

 Trainable params: 343,825 (1.31 MB)

 Non-trainable params: 0 (0.00 B)

## Compile the model

In [ ]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

## Train

In [ ]:
num_epochs = 10
history = model.fit(train_padded, train_labels,
                    epochs=num_epochs, verbose=1,
                    validation_split=0.1)

Epoch 1/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 38s 160ms/step - accuracy: 0.5093 - loss: 0.6936 - val_accuracy: 0.4933 - val_loss: 0.6933
Epoch 2/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 30s 140ms/step - accuracy: 0.5083 - loss: 0.6932 - val_accuracy: 0.5067 - val_loss: 0.6931
Epoch 3/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 42s 148ms/step - accuracy: 0.4935 - loss: 0.6934 - val_accuracy: 0.5067 - val_loss: 0.6931
Epoch 4/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 31s 146ms/step - accuracy: 0.4997 - loss: 0.6932 - val_accuracy: 0.4933 - val_loss: 0.6932
Epoch 5/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 31s 146ms/step - accuracy: 0.5033 - loss: 0.6932 - val_accuracy: 0.4933 - val_loss: 0.6932
Epoch 6/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 29s 138ms/step - accuracy: 0.5033 - loss: 0.6932 - val_accuracy: 0.4933 - val_loss: 0.6932
Epoch 7/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 30s 141ms/step - accuracy: 0.4953 - loss: 0.6931 - val_accuracy: 0.4933 - val_loss: 0.6932
Epoch 8/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 30s 142ms/step - accuracy: 0.5021 - loss: 0

## Test

In [ ]:
prediction = model.predict(test_padded)
# Get labels based on probability 1 if p>= 0.5 else 0
pred_labels = []
for i in prediction:
    if i >= 0.5:
        pred_labels.append(1)
    else:
        pred_labels.append(0)
print("Accuracy of prediction on test set : ", accuracy_score(test_labels,pred_labels))

79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step
Accuracy of prediction on test set :  0.502


## Working on New data

In [ ]:
sentence = ["The movie was very touching and heart whelming",
            "I have never seen a terrible movie like this",
            "the movie plot is terrible but it had good acting"]
# convert to a sequence
sequences = tokenizer.texts_to_sequences(sentence)
# pad the sequence
padded = pad_sequences(sequences, padding='post', maxlen=max_length)
# Get labels based on probability 1 if p>= 0.5 else 0
prediction = model.predict(padded)
pred_labels = []
for i in prediction:
    if i >= 0.5:
        pred_labels.append(1)
    else:
        pred_labels.append(0)
for i in range(len(sentence)):
    print(sentence[i])
    if pred_labels[i] == 1:
        s = 'Positive'
    else:
        s = 'Negative'
    print("Predicted sentiment : ",s)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
The movie was very touching and heart whelming
Predicted sentiment :  Negative
I have never seen a terrible movie like this
Predicted sentiment :  Negative
the movie plot is terrible but it had good acting
Predicted sentiment :  Negative
